# Week 7: Classic Features & Matching — Lab Practice

**Topics:** Harris Corner Detection, SIFT Keypoints & Descriptors, Descriptor Space, Feature Matching, RANSAC

This notebook accompanies the Week 7 lecture slides. Most operations are
one-line OpenCV calls — so the focus is on:

1. **Data structures** — what exactly do `cv2.SIFT_create().detectAndCompute()`
   and `cv2.findHomography()` return?
2. **Descriptor space** — a 128-dim SIFT descriptor is just a point in space;
   similarity = L2 proximity. We will compute this distance by hand.
3. **Parameter sensitivity** — Harris threshold, Lowe's ratio, RANSAC on/off
   all have strongly visible effects.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2

%matplotlib inline

### Environment setup

This notebook works both **locally** and on **Google Colab**.

In [ ]:
import os, urllib.request

# Detect Google Colab
IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if IN_COLAB:
    REPO_URL = "https://raw.githubusercontent.com/HyeongminLEE/image-processing-tutorial/main"
    os.makedirs("images", exist_ok=True)
    for fname in ["mot_color70.jpg", "mot_color83.jpg"]:
        url = f"{REPO_URL}/images/{fname}"
        if not os.path.exists(f"images/{fname}"):
            urllib.request.urlretrieve(url, f"images/{fname}")
            print(f"Downloaded {fname}")
    IMG_DIR = "images/"
else:
    IMG_DIR = "../images/"

print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Image directory: {IMG_DIR}")

### Display helpers

In [ ]:
def show_image(img, title=None, scale=5):
    """Display a single image. Grayscale arrays auto-use gray colormap."""
    fig, ax = plt.subplots(figsize=(scale, scale))
    if img.ndim == 2:
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(img)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_map(m, title=None, scale=5, cmap="gray", colorbar=False):
    """Display a single 2D float array with colormap."""
    fig, ax = plt.subplots(figsize=(scale, scale))
    im = ax.imshow(m, cmap=cmap)
    if colorbar:
        plt.colorbar(im, ax=ax, fraction=0.046)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_keypoints(img, keypoints, title=None, scale=7):
    """Draw SIFT-style keypoints (circle = scale, line = orientation)."""
    out = cv2.drawKeypoints(
        img, keypoints, None,
        flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS,
    )
    fig, ax = plt.subplots(figsize=(scale, scale))
    ax.imshow(out)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_matches(img1, kp1, img2, kp2, matches, title=None, scale=12):
    """Side-by-side visualization of matched keypoint pairs."""
    out = cv2.drawMatches(
        img1, kp1, img2, kp2, matches, None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )
    fig, ax = plt.subplots(figsize=(scale, scale / 2))
    ax.imshow(out)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

---

## 0. Why Features?

A **feature** (keypoint) is a point in an image that is distinctive enough
to be found again in a different view of the same scene. The full pipeline
we will build:

**detect** (find keypoints) → **describe** (summarize each one as a vector)
→ **match** (find pairs across two images) → **verify** (RANSAC to drop bad matches)

All four stages are one-line OpenCV calls — but each returns non-trivial
data structures. Learning to *read* those structures is the point of this lab.

---

## 1. Harris Corner Detection

Harris detects **corners** — points where intensity changes in every direction.
OpenCV's `cv2.cornerHarris` returns a **response map** `R` of the same shape
as the image. Peaks of `R` are corners.

In [ ]:
# Load the primary image pair for this week — two frames of the same street scene
img1 = np.array(Image.open(IMG_DIR + "mot_color70.jpg"))
img2 = np.array(Image.open(IMG_DIR + "mot_color83.jpg"))

gray1 = np.array(Image.open(IMG_DIR + "mot_color70.jpg").convert("L"))
gray2 = np.array(Image.open(IMG_DIR + "mot_color83.jpg").convert("L"))

print(f"img1 shape = {img1.shape}, dtype = {img1.dtype}")
print(f"gray1 shape = {gray1.shape}, dtype = {gray1.dtype}")

In [ ]:
show_image(img1, title="img1 (mot_color70)")
show_image(img2, title="img2 (mot_color83)")

`cv2.cornerHarris` wants `float32` input.
- `blockSize` — size of the neighborhood used to build the second-moment matrix `M`
- `ksize` — Sobel aperture for computing gradients
- `k` — Harris constant, typically 0.04

In [ ]:
R = cv2.cornerHarris(gray1.astype(np.float32), blockSize=2, ksize=3, k=0.04)

print(f"R shape = {R.shape}, dtype = {R.dtype}")
print(f"R range: [{R.min():.2e}, {R.max():.2e}]")

In [ ]:
# The response map itself — bright spots are corners, dark lines are edges
show_map(R, title="Harris response R", colorbar=True)

To **extract** corners, threshold `R` as a fraction of its maximum, then
overlay on the original image. `cv2.dilate` thickens the dots for visibility.

In [ ]:
corners_mask = R > 0.01 * R.max()
corners_mask = cv2.dilate(corners_mask.astype(np.uint8), None)

overlay = img1.copy()
overlay[corners_mask.astype(bool)] = [255, 0, 0]

show_image(overlay, title="Harris corners (threshold = 0.01 * R.max())")
print(f"number of corner pixels: {int(corners_mask.sum())}")

---

## 2. SIFT: Keypoints & Descriptors

Harris only gives locations `(x, y)`. SIFT goes further:
- **Detect** a keypoint at `(x, y, σ)` — position *and* scale
- Assign a **dominant orientation** `θ`
- Emit a **128-dim descriptor** summarizing the local gradient pattern

In [ ]:
sift = cv2.SIFT_create()
kp, des = sift.detectAndCompute(gray1, None)

print(f"number of keypoints = {len(kp)}")
print(f"descriptors shape   = {des.shape}")
print(f"descriptors dtype   = {des.dtype}")

Each element of `kp` is a `cv2.KeyPoint` — inspect one directly:

In [ ]:
k0 = kp[0]
print(f"kp[0].pt    = {k0.pt}       # (x, y) location")
print(f"kp[0].size  = {k0.size:.2f}  # scale (diameter in pixels)")
print(f"kp[0].angle = {k0.angle:.2f} # dominant orientation in degrees")

Visualize with `DRAW_RICH_KEYPOINTS` — the circle radius encodes scale
and the radial line encodes orientation.

In [ ]:
show_keypoints(img1, kp, title=f"{len(kp)} SIFT keypoints (rich)")

---

## 3. Descriptor Space

Each row of `des` is a **point in 128-dimensional space**. Similar patches
map to nearby points; different patches map to distant points. Matching is
literally "find the nearest neighbor in this space."

L2 distance measures how close two descriptors are — sum the squared differences, then take the square root:

$$d(a, b) = \|a - b\|_2$$

In [ ]:
d_self = np.linalg.norm(des[0] - des[0])
d_rand = np.linalg.norm(des[0] - des[100])
print(f"||des[0] - des[0]||   = {d_self:.2f}  # identical descriptors")
print(f"||des[0] - des[100]|| = {d_rand:.2f}  # two random keypoints in the same image")

Now compute SIFT on **both** images so we can search across them.

In [ ]:
kp1, des1 = sift.detectAndCompute(gray1, None)
kp2, des2 = sift.detectAndCompute(gray2, None)

print(f"image 1: {len(kp1)} keypoints,  des1 shape = {des1.shape}")
print(f"image 2: {len(kp2)} keypoints,  des2 shape = {des2.shape}")

---

## 4. Feature Matching with Lowe's Ratio Test

OpenCV's **BFMatcher** does brute-force L2 nearest-neighbor search: for every descriptor in `des1`, find the closest descriptor in `des2`.

In [ ]:
# Brute-force matcher with L2 distance — the same norm we saw in Section 3
bf = cv2.BFMatcher(cv2.NORM_L2)

# k=2 → return the top-2 nearest neighbors for each query descriptor
matches = bf.knnMatch(des1, des2, k=2)

print(f"raw matches = {len(matches)}  (one pair (m, n) per query descriptor)")
print(f"example: matches[0][0].distance = {matches[0][0].distance:.2f}")
print(f"         matches[0][1].distance = {matches[0][1].distance:.2f}")

**Lowe's ratio test**: keep a match only if the best is *clearly* better
than the second-best.

$$\frac{d_1}{d_2} < 0.7$$

Ambiguous queries (where $d_1 \approx d_2$) get discarded.

In [ ]:
good_07 = [m for m, n in matches if m.distance < 0.7 * n.distance]
print(f"good matches at ratio 0.7: {len(good_07)}  (out of {len(matches)})")

show_matches(img1, kp1, img2, kp2, good_07, title="ratio = 0.7")

### Exercises

**Exercise 4.1:** Sweep the ratio threshold.

Compute `good` matches at three thresholds: **0.5**, **0.7**, **0.9**.
For each, print the count and visualize with `show_matches`.

*Hint:* Three explicit list comprehensions — one per threshold — with the
same `(m, n) in matches` pattern used above.

In [ ]:
# Input — matches already computed above
print(f"raw matches = {len(matches)}")

# YOUR CODE HERE
# Produce three filtered lists using Lowe's ratio test
# (see the ratio=0.7 line in the Section 4 demo for the pattern):
#   good_05 — matches at ratio threshold 0.5
#   good_07 — matches at ratio threshold 0.7
#   good_09 — matches at ratio threshold 0.9
# Print the length of each.


# --- Output display ---
show_matches(img1, kp1, img2, kp2, good_05, title="ratio = 0.5 (strict)")
show_matches(img1, kp1, img2, kp2, good_07, title="ratio = 0.7")
show_matches(img1, kp1, img2, kp2, good_09, title="ratio = 0.9 (loose)")


# How does the number of matches — and the number of *visibly wrong* matches —
# change as the ratio increases from 0.5 to 0.9?
# Which threshold would you use for panorama stitching (few but clean matches)?
# (You may write your answer in Korean.)
#

---

## 5. RANSAC: Robust Homography Estimation

Even after the ratio test, some matches are wrong. A single outlier can
**corrupt** the homography `H` if we fit it naively. RANSAC repeatedly
samples 4 pairs, fits a tentative `H`, and counts inliers — the model
with the most inliers wins.

First extract numeric point coordinates from the `good_07` match objects:

In [ ]:
# Reuse the 0.7 ratio matches from Section 4
good = good_07

src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)

print(f"{len(good)} correspondences")
print(f"src_pts shape = {src_pts.shape}")

`cv2.findHomography` with `method=cv2.RANSAC` returns `H` **plus an inlier
mask**: `mask[i] == 1` means match `i` agreed with the winning model.

In [ ]:
H_ransac, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
inliers = int(mask.sum())
print(f"H_ransac =\n{np.round(H_ransac, 4)}")
print(f"inliers = {inliers} / {len(mask)}")

# Keep only the inlier matches, then visualize
inlier_good = [m for m, keep in zip(good, mask.ravel()) if keep]
show_matches(img1, kp1, img2, kp2, inlier_good, title=f"RANSAC inliers ({inliers})")

### Exercises

**Exercise 5.1:** RANSAC on vs. off — warp comparison.

The naive homography (`method=0`) fits **all** good matches equally, so a
few outliers can ruin it. Compare visually:

1. Compute `H_naive = cv2.findHomography(src_pts, dst_pts, 0)[0]` — no RANSAC
2. (`H_ransac` is already computed above.)
3. Warp `img1` with each to the shape `(W, H) = (img2.shape[1], img2.shape[0])`
   using `cv2.warpPerspective`.
4. Display `img2` (reference), `warp_naive`, and `warp_ransac`.

*Hint:* `cv2.warpPerspective(img, H, (W, H_img))` — note the `(width, height)`
order. `img2.shape[:2]` gives `(H_img, W)` — reverse it.

In [ ]:
# Input
show_image(img1, title="img1 (source)")
show_image(img2, title="img2 (reference — where we want img1 to land)")

# YOUR CODE HERE
# 1. H_naive — same findHomography call as the RANSAC demo above, but pass
#              method=0 instead of cv2.RANSAC (no RANSAC, fits ALL good matches).
# 2. warp_naive  — warp img1 by H_naive  using cv2.warpPerspective
# 3. warp_ransac — warp img1 by H_ransac using cv2.warpPerspective
#    Signature: cv2.warpPerspective(img, H, (width, height))
#    Use img2's size: width = img2.shape[1], height = img2.shape[0].


# --- Output display ---
show_image(img2,        title="img2 (reference)")
show_image(warp_naive,  title="warp with H_naive (no RANSAC)")
show_image(warp_ransac, title="warp with H_ransac")


# How different are the two warps? One outlier can shift the entire
# estimated plane — is the difference visible?
# (You may write your answer in Korean.)
#

---

## Summary

- **Harris** — second-moment-matrix response map → threshold → corners
- **SIFT** — `(x, y, σ, θ)` keypoints + 128-dim descriptors
- **Descriptor space** — a descriptor is a point; matching = nearest neighbor in L2
- **Lowe's ratio test** — keep a match only if $d_1 / d_2 < 0.7$
- **RANSAC** — one outlier can ruin `H`; random sampling + consensus saves us

**Next (Week 9):** we hand-crafted 128 dimensions. Can a neural network
learn a better descriptor?